In [ ]:
import csv
from pathlib import Path
import pandas as pd

def _load_csv(file_path: Path) -> pd.DataFrame:
    """
    Tenta inferir o delimitador e o encoding de um arquivo CSV 
    antes de carregá-lo em um DataFrame.
    """
    encodings_para_testar = ['utf-8', 'latin1', 'iso-8859-1', 'cp1252']
    
    for encoding in encodings_para_testar:
        try:
            # Lê apenas a primeira linha para inferir o delimitador
            with open(file_path, 'r', encoding=encoding) as f:
                amostra = f.readline()
                try:
                    delimitador = csv.Sniffer().sniff(amostra).delimiter
                except csv.Error:
                    # Se o sniffer falhar (ex: arquivo de 1 coluna), assume vírgula
                    delimitador = ','
            
            # Se conseguiu ler e achar o delimitador, carrega o DataFrame
            df = pd.read_csv(file_path, sep=delimitador, encoding=encoding)
            return df
            
        except UnicodeDecodeError:
            # Se der erro de encoding, ignora e tenta o próximo da lista
            continue
            
    # Se testou todos os encodings e não conseguiu, levanta um erro
    raise ValueError(
        f"Não foi possível ler o arquivo {file_path}. "
        f"Encodings tentados: {encodings_para_testar}"
    )

def load_data(file_path: str) -> pd.DataFrame:
    """
    Carrega um arquivo de dados (.csv ou .xlsx) e retorna um pandas DataFrame.
    Valida a existência do arquivo e se a extensão é suportada.
    """
    path = Path(file_path)
    
    # 1. Verifica se o arquivo existe
    if not path.exists():
        raise FileNotFoundError(f"Erro: O arquivo '{file_path}' não foi encontrado.")
        
    extensao = path.suffix.lower()
    
    # 2 e 3. Verifica extensão e carrega os dados
    if extensao == '.csv':
        return _load_csv(path)
    elif extensao == '.xlsx':
        # openpyxl é usado automaticamente pelo pandas por baixo dos panos para xlsx
        return pd.read_excel(path)
    else:
        raise ValueError(
            f"Extensão '{extensao}' não suportada. "
            "Utilize apenas arquivos .csv ou .xlsx."
        )
    